# Phases 3–5: Validation design, modelling and evaluation

* `src/mirageai/cv.py`: variogram, spatial blocks, random / spatial / spatio-temporal splitters
* `src/mirageai/models.py`: five models, search spaces, tuning inside spatial CV
* `src/mirageai/evaluate.py`: metrics, CV gap, temporal holdout, residual diagnostics

The heavy lifting runs in `scripts/pipeline/run_phase3.py`, `scripts/pipeline/run_phase4.py` (≈30 min) and `scripts/pipeline/run_phase5.py`.
This notebook reads their outputs.

In [1]:
import pandas as pd
from mirageai import config, results as R

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
from mirageai import evaluate as E

## Block size from the residual variogram (D-028)

![Variogram](../outputs/figures/03_variogram_lst_anom.png)

In [2]:
R.load_table("phase4_residual_range")

,year,resid_var,nugget,sill,range_90_m,range_95_m,range_99_m,target_var
0,2023,3.336,1.292,3.239,2550.0,5050.0,6950.0,13.481
1,2024,2.747,1.249,2.631,2150.0,3950.0,5950.0,10.424
2,2025,2.905,1.294,2.882,1450.0,2350.0,5650.0,11.355


## Model × scheme comparison (mean over 5 layouts × 5 folds)

In [3]:
comp = R.load_table("model_comparison")
comp[comp.scheme.isin(["random", E.PRIMARY_SCHEME, E.SPATIO_TEMPORAL_SCHEME])][
    ["scheme", "model", "n_features", "RMSE_mean", "RMSE_std", "MAE_mean", "R2_mean"]]

,scheme,model,n_features,RMSE_mean,RMSE_std,MAE_mean,R2_mean
0,random,baseline,12,3.428,0.015,2.777,-6.030e-05
1,random,linear,12,1.849,0.010,1.461,7.090e-01
2,random,rf,13,1.637,0.008,1.291,7.719e-01
3,random,xgb,13,1.594,0.009,1.256,7.837e-01
4,random,lgbm,13,1.620,0.009,1.278,7.768e-01
15,spatial_4km,baseline,12,3.451,0.187,2.801,-4.202e-02
16,spatial_4km,linear,12,1.873,0.047,1.480,6.909e-01
17,spatial_4km,rf,13,1.756,0.059,1.382,7.284e-01
18,spatial_4km,xgb,13,1.746,0.064,1.373,7.315e-01
19,spatial_4km,lgbm,13,1.746,0.060,1.375,7.316e-01


## RQ3: how much random CV flatters each model

In [4]:
R.block_size_curve(comp).pivot(index="block_km", columns="model", values="RMSE_mean")

model,baseline,lgbm,linear,rf,xgb
block_km,,,,,
0,3.428,1.620,1.849,1.637,1.594
1,3.428,1.675,1.853,1.693,1.665
2,3.435,1.706,1.859,1.722,1.702
4,3.451,1.746,1.873,1.756,1.746
8,3.490,1.840,1.910,1.842,1.849


![CV gap](../outputs/figures/05_cv_gap.png)

![Residual map](../outputs/figures/05_residual_map.png)

## Where the errors are: hottest-decile detection (chance = 10%)

In [5]:
R.hot_decile_detection(R.load_table("phase5_hot_decile_overlap"))

,model,decile,mean,std,model_label
0,baseline,AOI-wide,0.165,0.030,Mean baseline
1,baseline,within urban,0.164,0.067,Mean baseline
2,lgbm,AOI-wide,0.623,0.025,LightGBM
3,lgbm,within urban,0.542,0.013,LightGBM
4,linear,AOI-wide,0.590,0.025,Linear regression
5,linear,within urban,0.489,0.019,Linear regression
6,rf,AOI-wide,0.619,0.026,Random forest
7,rf,within urban,0.534,0.013,Random forest
8,xgb,AOI-wide,0.625,0.025,XGBoost
9,xgb,within urban,0.545,0.013,XGBoost
